# Colab 29b — Unified comparison + ProtTrans: method × metric × alphabet

*Fork of colab29 with **ProtTrans/ProtT5 promoted to a live second PLM baseline** next to ESM-2, folded into every table and figure. See `CONTINUE_v20_prottrans_baseline.md`.*

**One harness, one shared eval, apples-to-apples where the sampling unit matches the question.** Every CATH
method is scored on the *same pool and the same pairs* (per feed) under the same three metrics, so the deck
can show a method × metric table instead of stitching numbers across notebooks. The synthetic AA column is
an **in-distribution reference** for pairwise geometry/discrimination, not a retrieval pool.

- **Methods (rows, presentation order):** **SNN**, ESM2, **ProtTrans**, Dice, trigram-count, length-only
  floor. **ProtTrans/ProtT5** = second protein-LM baseline, live here, parity-pooled with ESM-2 and kept
  next to it so the two learned PLMs group together and the classical baselines follow.
- **Columns for Spearman/AUROC:** AA synthetic / 3Di / SS / AA CATH_s20.
- **Metrics:** **Spearman rho(sim, normLev)** — primary, threshold-free, on a *stratified full-range*
  pair set; **AUROC** (is-high >= 0.70 vs background); **MAP@10 / hit@10** (CATH full-pool exhaustive
  de-hubbed oracle, bars 0.70 and 0.90, with the length floor).

Ground truth is **exact normLev on our own strings** only (Fenoy's 0.66 is vs BLAST — context, not a direct
comparison). Add a method = add one entry to `METHODS`; all three CATH metrics + figures run over it.

*This is a NEW results-of-record run (adds ProtT5, recomputes every method under one shared protocol). It does
NOT inherit the July 14/16 colab29 snapshot — all outputs use the `colab29b_*` namespace, and a fresh dated
receipt (`RESULTS_colab29b_<date>.md`) should be written after running, verifying SNN/ESM2/classical values
against the prior run before any slide is updated. D1: one frozen AA-trained SNN encoder scores AA / SS / 3Di /
synthetic AA; the 30k synthetic training set is REGENERATED each run (seeded, not persisted) — record that in
the receipt so a delta can be attributed to a new baseline vs a changed training set.*


## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
CACHE = '/content/bench_cache'; os.makedirs(CACHE, exist_ok=True)
for f in ['cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz',
          'cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy transformers sentencepiece matplotlib --quiet

In [ ]:
import time, json, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.sparse as sp
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED); rng = np.random.default_rng(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
# --- record dependency versions for the results receipt (unpinned installs) ---
import transformers, scipy, sklearn, rapidfuzz
print('versions | torch', torch.__version__, '| transformers', transformers.__version__,
      '| numpy', np.__version__, '| scipy', scipy.__version__,
      '| sklearn', sklearn.__version__, '| rapidfuzz', rapidfuzz.__version__)
try:
    import sentencepiece; print('versions | sentencepiece', sentencepiece.__version__)
except Exception as _e:
    print('versions | sentencepiece NOT importable:', _e)

# ---- toggles ----
ENABLE_PROTTRANS = True    # ProtT5 live as a SECOND PLM baseline next to ESM-2 (needs GPU + sentencepiece)
PROT_TRIM_EOS    = True    # pool residues only (drops the appended </s>) = ProtT5 model-card recipe + ESM-2 parity.
                           #   set False to mean-pool the RAW attention mask incl. </s> (A/B check; effect is tiny for L>>1)
N_STRAT_PER_BIN  = 400     # target pairs per normLev decile bin, per feed (capped by availability)
N_STRAT_CAND     = 300_000 # random candidate pairs sampled per feed before binning
N_BOOT           = 1000    # bootstrap resamples for MAP@10 CI
BOOT_SEED        = 20260723 # dedicated bootstrap RNG (order-independent, decoupled from train/sampling `rng`)

## 2. Constants + helpers + SNN architecture (colab24e/28 recipe, verbatim)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN = 50, 200; N_TRAIN, EPOCHS, BS, K = 30000, 30, 128, 16
BAND_LOW_AA, BAND_LOW_SS, BAND_HIGH, BAND_STRICT = 0.30, 0.56, 0.70, 0.90
BAND_MID = 0.30   # lower edge of the hard-negative window [0.30, 0.70) for full-pool AUROC
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)
def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L
def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx, dtype=torch.long)
def perturb(seq, k, abc):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)
def rand_seq(abc): L = int(rng.integers(MIN_LEN, MAX_LEN+1)); return ''.join(rng.choice(list(abc), size=L))
def bin_idx(x, band_low): return 0 if x < band_low else (1 if x < BAND_HIGH else 2)

In [ ]:
class Enc(nn.Module):
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64*K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)
class Clf(nn.Module):
    def __init__(s):
        super().__init__(); s.encoder = Enc()
        s.head = nn.Sequential(nn.Linear(128, 64), nn.LeakyReLU(0.01), nn.Linear(64, 3))
    def forward(s, a, b): return s.head(torch.abs(s.encoder(a) - s.encoder(b)))
class DS(Dataset):
    def __init__(s, pp, bl): s.p = pp; s.bl = bl
    def __len__(s): return len(s.p)
    def __getitem__(s, i): a, b, l = s.p[i]; return encode_pad(a), encode_pad(b), bin_idx(l, s.bl)

def train_snn(alphabet, band_low, label):
    t0 = time.perf_counter(); pairs = []
    while len(pairs) < N_TRAIN:
        sd = rand_seq(alphabet); L = len(sd); t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, alphabet)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    datagen_s = time.perf_counter() - t0
    dl = DataLoader(DS(pairs, band_low), batch_size=BS, shuffle=True)
    model = Clf().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3); crit = nn.CrossEntropyLoss()
    model.train(); t0 = time.perf_counter()
    for ep in range(1, EPOCHS+1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1: print(f'  [{label}] epoch {ep}/{EPOCHS} CE {tot/nb:.4f}')
    if device.type == 'cuda': torch.cuda.synchronize()
    model.eval(); return model, datagen_s, time.perf_counter() - t0
snn_params = sum(p.numel() for p in Enc().parameters())
print(f'SNN encoder params = {snn_params:,}')

## 3. Per-representation pools (colab24f/28 protocol, verbatim)

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')
RESCUED = {'4z0mC02', '3qkaE02'}
def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq) and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])            if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])            if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}
LOOK = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_IDS = {f: list(LOOK[f]) for f in LOOK}
POOL_SEQ = {f: [LOOK[f][d] for d in POOL_IDS[f]] for f in LOOK}
POOL_LEN = {f: np.array([len(s) for s in POOL_SEQ[f]]) for f in LOOK}
FEEDS = ['AA', 'SS', '3Di']
for f in FEEDS: print(f'  {f:<4} pool = {len(POOL_IDS[f]):>6}')

## 4. Train the ONE SNN encoder (frozen afterwards) — AA-trained, used for all three feeds

**Locked 2026-07-13 (D1):** a *single* encoder, trained once on synthetic uniform-AA pairs, is evaluated
zero-shot on AA / SS / 3Di. The old SS-trained encoder is dropped: it made the SS column a *retrained*
result while AA/3Di were transfer results, which muddled the cross-alphabet claim. One encoder = every
SNN cell in every table is the same frozen model, so Q2 is a clean transfer statement.

*(SS/3Di strings tokenise through the AA vocabulary — both use AA-letter alphabets. `BAND_LOW_SS` is now
unused for training and kept only for reference.)*

In [ ]:
model_aa, _, _ = train_snn(AA_ALPHABET, BAND_LOW_AA, 'AA')
SNN_BY_FEED = {'AA': model_aa, 'SS': model_aa, '3Di': model_aa}   # D1: one AA-trained encoder, zero-shot on all feeds
print('One encoder (AA-trained) scores all three feeds — every SNN number below is a transfer result.')

## 5. Exhaustive Levenshtein oracle (de-hubbed true-neighbour sets, bars 0.70 & 0.90)

Same all-pairs exhaustive protocol as colab24f: full pool × pool exact Levenshtein, blocked. Returns
true-neighbour sets at each bar (self excluded) and, as a by-product, every high-sim positive pair
(used to guarantee the top normLev bins are populated for the Spearman set, and as AUROC positives).

In [ ]:
def build_oracle(feed, block=1024):
    seqs = POOL_SEQ[feed]; lens = POOL_LEN[feed]; N = len(seqs)
    T_high, T_strict = {}, {}
    pos_pairs = []   # (i, j, normLev) for j>i with normLev >= BAND_HIGH  (Spearman top-bin + AUROC pos)
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - Dm / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0
            hi = np.where(row >= BAND_HIGH)[0]
            if hi.size: T_high[i] = hi.astype(np.int32)
            st = np.where(row >= BAND_STRICT)[0]
            if st.size: T_strict[i] = st.astype(np.int32)
            for j in hi:
                if j > i: pos_pairs.append((i, int(j), float(row[j])))
    med_h = int(np.median([len(v) for v in T_high.values()])) if T_high else 0
    print(f'  oracle[{feed}]: queries@0.70={len(T_high)} (med|T|={med_h}), '
          f'queries@0.90={len(T_strict)}, high-sim pos pairs={len(pos_pairs)}')
    return dict(T_high=T_high, T_strict=T_strict, pos_pairs=pos_pairs)
print('Building exact-Levenshtein oracle (exhaustive, de-hubbed)...')
ORACLE = {f: build_oracle(f) for f in FEEDS}

## 6. Shared stratified pair set — per feed, by that feed's OWN normLev

For each feed: sample many random pool pairs, compute normLev, bin into deciles [0,.1)…[.9,1], and
draw up to `N_STRAT_PER_BIN` per populated bin. Guaranteed to include the exhaustive high-sim pairs so
the top bins are covered even for AA (where natural high-sim pairs are extremely rare). **This exact
pair set is reused by every method** so the Spearman/AUROC comparison is fair. AA's top bins will be
sparse by construction — that scarcity is the data reality, reported, not hidden.

In [ ]:
def build_strat_pairs(feed):
    N = len(POOL_SEQ[feed]); seqs = POOL_SEQ[feed]
    # 1) random candidate pairs
    a = rng.integers(0, N, size=N_STRAT_CAND); b = rng.integers(0, N, size=N_STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    # 2) add exhaustive high-sim positives (guarantees top-bin coverage) — build once, concat once
    pp = ORACLE[feed]['pos_pairs']
    if pp:
        parr = np.array(pp, dtype=float)   # (P,3): i, j, normLev
        a = np.concatenate([a, parr[:, 0].astype(np.int64)])
        b = np.concatenate([b, parr[:, 1].astype(np.int64)])
        nl = np.concatenate([nl, parr[:, 2]])
    # 3) stratify into deciles
    edges = np.linspace(0.0, 1.0, 11)
    bins = np.clip(np.digitize(nl, edges) - 1, 0, 9)
    ai, aj, av, binlist = [], [], [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size == 0: continue
        take = rng.permutation(idx)[:N_STRAT_PER_BIN]
        ai.append(a[take]); aj.append(b[take]); av.append(nl[take]); binlist.append(np.full(take.size, bb))
    ai = np.concatenate(ai); aj = np.concatenate(aj); av = np.concatenate(av); binlist = np.concatenate(binlist)
    return dict(i=ai.astype(np.int64), j=aj.astype(np.int64), nl=av, bins=binlist)

STRAT = {f: build_strat_pairs(f) for f in FEEDS}
print('Stratified pair set per feed (count per normLev decile):')
hdr = 'feed  ' + ''.join(f'[{k/10:.1f})'.rjust(7) for k in range(10)) + '   total'
print(hdr); print('-'*len(hdr))
for f in FEEDS:
    counts = [int((STRAT[f]['bins'] == b).sum()) for b in range(10)]
    print(f'{f:<5} ' + ''.join(str(c).rjust(7) for c in counts) + f'   {sum(counts):>6}')

## 7. Method registry — add a method = add one entry

Each method exposes two vectorized calls over a feed's pool:
- `pair(feed, i, j)` → similarity for pair arrays (used by Spearman + AUROC),
- `query(feed, qi)` → similarity of pool item `qi` vs the whole pool (used by retrieval; self set to −∞).

Embedding methods (SNN, ESM2, ProtTrans) share one implementation over an L2-normalised pool matrix
(cosine = dot). Spearman/AUROC/retrieval are all rank-based, so cosine is metric-equivalent to the
SNN's `1−L2/2` convention. String methods (trigram/Dice) use a sparse binary k-mer matrix. Length is
the trivial floor.

In [ ]:
import os, re, hashlib
def _seqsig(seqs):
    return hashlib.sha256('\x00'.join(seqs).encode()).hexdigest()[:16]  # invalidate cache if pool sequences change under fixed ids
# ---------- embedding backends ----------
@torch.no_grad()
def snn_pool(feed):
    ids = POOL_IDS[feed]; look = LOOK[feed]; model = SNN_BY_FEED[feed]
    out = np.zeros((len(ids), 128), np.float32); model.eval()
    for i in range(0, len(ids), 256):
        x = torch.stack([encode_pad(look[d]) for d in ids[i:i+256]]).to(device)
        out[i:i+x.shape[0]] = model.encoder(x).cpu().numpy()
    return out

ESM2_MODEL = 'facebook/esm2_t12_35M_UR50D'
_esm = {}
def _load_esm():
    if 'mdl' in _esm: return
    from transformers import AutoTokenizer, AutoModel
    _esm['tok'] = AutoTokenizer.from_pretrained(ESM2_MODEL)
    _esm['mdl'] = AutoModel.from_pretrained(ESM2_MODEL).to(device).eval()
@torch.no_grad()
def esm_pool(feed):
    _load_esm(); tok, mdl = _esm['tok'], _esm['mdl']
    cf = f'{CACHE}/esm2_29_{feed}.npy'; sf = f'{CACHE}/esm2_29_{feed}.meta.json'
    ids = POOL_IDS[feed]; seqs = POOL_SEQ[feed]; sig = _seqsig(seqs)
    if os.path.exists(cf) and os.path.exists(sf):
        meta = json.load(open(sf))
        if meta.get('model') == ESM2_MODEL and meta.get('ids') == ids and meta.get('sig') == sig: return np.load(cf)
    order = np.argsort([len(s) for s in seqs]); out = [None]*len(seqs)
    for i in range(0, len(order), 32):
        idx = order[i:i+32]; batch = [seqs[j] for j in idx]
        enc = tok(batch, return_tensors='pt', padding=True, add_special_tokens=True).to(device)
        h = mdl(**enc).last_hidden_state; mask = enc['attention_mask'].clone(); mask[:, 0] = 0
        for r, l in enumerate(enc['attention_mask'].sum(1)): mask[r, l-1] = 0
        m = mask.unsqueeze(-1).float(); e = (h*m).sum(1)/m.sum(1).clamp(min=1)
        e = F.normalize(e, dim=1).cpu().numpy()
        for kk, j in enumerate(idx): out[j] = e[kk]
    e = np.stack(out).astype(np.float32); np.save(cf, e)
    json.dump({'model': ESM2_MODEL, 'ids': ids, 'sig': sig}, open(sf, 'w')); return e

PROTTRANS_MODEL = 'Rostlab/prot_t5_xl_half_uniref50-enc'
_pt = {}
def _load_prot():
    if 'mdl' in _pt: return
    from transformers import T5Tokenizer, T5EncoderModel
    _pt['tok'] = T5Tokenizer.from_pretrained(PROTTRANS_MODEL, do_lower_case=False)
    _pt['mdl'] = T5EncoderModel.from_pretrained(PROTTRANS_MODEL).to(device).eval()
@torch.no_grad()
def prot_pool(feed):
    # Second PLM baseline (ProtT5-XL, UniRef50). Pooled to PARITY with esm_pool:
    #  - cache to CACHE (point CACHE at Drive so reruns skip the ~1.2B-param embed),
    #  - pool residues only (drop trailing </s>) per the ProtT5 model-card recipe [:seq_len].mean(),
    #    matching ESM-2's BOS/EOS handling -- toggle PROT_TRIM_EOS=False to pool the raw mask instead,
    #  - defensive U/Z/O/B -> X (ProtT5's rare-AA convention; a no-op on our 20-letter feeds).
    tag = 'trim' if PROT_TRIM_EOS else 'full'
    cf = f'{CACHE}/prot_t5_29_{feed}_{tag}.npy'; sf = f'{CACHE}/prot_t5_29_{feed}_{tag}.meta.json'
    ids = POOL_IDS[feed]; seqs = POOL_SEQ[feed]; sig = _seqsig(seqs)
    if os.path.exists(cf) and os.path.exists(sf):
        meta = json.load(open(sf))
        if meta.get('model') == PROTTRANS_MODEL and meta.get('ids') == ids and meta.get('sig') == sig: return np.load(cf)
    _load_prot(); tok, mdl = _pt['tok'], _pt['mdl']
    prepped = [' '.join(re.sub(r'[UZOB]', 'X', s)) for s in seqs]   # ProtT5 spacing + rare-AA map
    order = np.argsort([len(s) for s in seqs]); out = [None]*len(seqs)   # length-sorted -> less padding
    for i in range(0, len(order), 16):
        idx = order[i:i+16]; batch = [prepped[j] for j in idx]
        enc = tok(batch, return_tensors='pt', padding=True, add_special_tokens=True).to(device)
        h = mdl(**enc).last_hidden_state; mask = enc['attention_mask'].clone()
        if PROT_TRIM_EOS:
            for r, l in enumerate(enc['attention_mask'].sum(1)): mask[r, l-1] = 0   # drop trailing </s> (EOS)
        m = mask.unsqueeze(-1).float(); e = (h*m).sum(1)/m.sum(1).clamp(min=1)
        e = F.normalize(e, dim=1).cpu().numpy()
        for kk, j in enumerate(idx): out[j] = e[kk]
    e = np.stack(out).astype(np.float32); np.save(cf, e)
    json.dump({'model': PROTTRANS_MODEL, 'ids': ids, 'sig': sig, 'trim_eos': PROT_TRIM_EOS}, open(sf, 'w'))
    # To free VRAM after all feeds (ESM2 + ProtT5 co-resident fits a T4/L4, so off by default):
    #   del _pt['mdl'], _pt['tok']; torch.cuda.empty_cache()
    return e

# ---------- sparse k-mer backend (trigram / Dice) ----------
def build_kmer(feed, k=3):
    seqs = POOL_SEQ[feed]; vocab = {}; rows, cols = [], []
    for r, s in enumerate(seqs):
        seen = set(s[t:t+k] for t in range(len(s)-k+1))
        for g in seen:
            c = vocab.setdefault(g, len(vocab)); rows.append(r); cols.append(c)
    B = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                      shape=(len(seqs), max(1, len(vocab))))
    sizes = np.asarray(B.sum(1)).ravel()
    return B, sizes

# ---------- caches ----------
EMB = {}      # (backend, feed) -> pool matrix
KMER = {}     # feed -> (B, sizes)
def get_emb(backend, feed):
    key = (backend, feed)
    if key not in EMB:
        EMB[key] = {'snn': snn_pool, 'esm': esm_pool, 'prot': prot_pool}[backend](feed)
    return EMB[key]
def get_kmer(feed):
    if feed not in KMER: KMER[feed] = build_kmer(feed)
    return KMER[feed]

In [ ]:
# ---------- unified similarity interface ----------
# Each method exposes: pair(feed,i,j) [Spearman], query(feed,qi) [retrieval, self=-inf],
# block(feed,r0,r1) [full-pool AUROC: rows r0:r1 vs all cols, shape (r1-r0, N)].
def _emb_pair(be, feed, i, j):
    E = get_emb(be, feed); return np.sum(E[i]*E[j], axis=1)
def _emb_query(be, feed, qi):
    E = get_emb(be, feed); s = E @ E[qi]; s[qi] = -np.inf; return s
def _emb_block(be, feed, r0, r1):
    E = get_emb(be, feed); return E[r0:r1] @ E.T

def _kmer_pair(feed, i, j, mode):
    B, sz = get_kmer(feed)
    inter = np.asarray(B[i].multiply(B[j]).sum(1)).ravel().astype(float); sa, sb = sz[i], sz[j]
    if mode == 'count': return inter                                    # shared 3-gram COUNT (literal)
    if mode == 'dice':  return 2*inter/np.maximum(sa+sb, 1e-9)          # Dice (length-fair)
    return inter/np.maximum(sa+sb-inter, 1e-9)                          # Jaccard (optional)
def _kmer_query(feed, qi, mode):
    B, sz = get_kmer(feed)
    inter = np.asarray(B.dot(B[qi].T).todense()).ravel().astype(float); sq = sz[qi]
    if mode == 'count': s = inter
    elif mode == 'dice': s = 2*inter/np.maximum(sz+sq, 1e-9)
    else: s = inter/np.maximum(sz+sq-inter, 1e-9)
    s[qi] = -np.inf; return s
def _kmer_block(feed, r0, r1, mode):
    B, sz = get_kmer(feed)
    C = np.asarray(B[r0:r1].dot(B.T).todense()).astype(float); a = sz[r0:r1][:, None]; b = sz[None, :]
    if mode == 'count': return C
    if mode == 'dice':  return 2*C/np.maximum(a+b, 1e-9)
    return C/np.maximum(a+b-C, 1e-9)

def _len_pair(feed, i, j):
    L = POOL_LEN[feed]; return 1.0 - np.abs(L[i]-L[j])/np.maximum(np.maximum(L[i], L[j]), 1)
def _len_query(feed, qi):
    L = POOL_LEN[feed]; s = 1.0 - np.abs(L-L[qi])/np.maximum(np.maximum(L, L[qi]), 1)
    s = s.astype(float); s[qi] = -np.inf; return s
def _len_block(feed, r0, r1):
    L = POOL_LEN[feed]; a = L[r0:r1][:, None]; b = L[None, :]
    return 1.0 - np.abs(a-b)/np.maximum(np.maximum(a, b), 1)

# name -> dict(pair, query, block, status). Both string baselines are TRIGRAM (3-gram) measures, as
# the professor asked: 'trigram' = COUNT of shared 3-grams |A∩B| (the literal ask; note it is
# length-biased — longer sequences share more); 'Dice' = 2|A∩B|/(|A|+|B|), the length-fair trigram
# comparator that keeps the SNN honest. (A Jaccard variant is one flag away: mode='jaccard'.)
METHODS = {
    'trigram':   dict(pair=lambda f,i,j:_kmer_pair(f,i,j,'count'), query=lambda f,q:_kmer_query(f,q,'count'), block=lambda f,r0,r1:_kmer_block(f,r0,r1,'count'), status='live'),
    'Dice':      dict(pair=lambda f,i,j:_kmer_pair(f,i,j,'dice'),  query=lambda f,q:_kmer_query(f,q,'dice'),  block=lambda f,r0,r1:_kmer_block(f,r0,r1,'dice'),  status='live'),
    'length':    dict(pair=_len_pair,                              query=_len_query,                          block=_len_block,                                  status='live'),
    'ESM2':      dict(pair=lambda f,i,j:_emb_pair('esm',f,i,j),    query=lambda f,q:_emb_query('esm',f,q),    block=lambda f,r0,r1:_emb_block('esm',f,r0,r1),    status='live'),
    'SNN':       dict(pair=lambda f,i,j:_emb_pair('snn',f,i,j),    query=lambda f,q:_emb_query('snn',f,q),    block=lambda f,r0,r1:_emb_block('snn',f,r0,r1),    status='live'),
    'ProtTrans': dict(pair=lambda f,i,j:_emb_pair('prot',f,i,j),   query=lambda f,q:_emb_query('prot',f,q),   block=lambda f,r0,r1:_emb_block('prot',f,r0,r1),   status='live'),
}
ACTIVE = [m for m, d in METHODS.items() if d['status'] == 'live' and (m != 'ProtTrans' or ENABLE_PROTTRANS)]
print('Active methods:', ACTIVE)
print('WIP (skipped, shown as WIP in tables):', [m for m in METHODS if m not in ACTIVE])

## 8. Metric 1 — Spearman ρ(sim, normLev)  [PRIMARY, centerpiece]

Threshold-free, whole-range, on the shared stratified pair set (per feed). Rank-based, so cosine vs
`1−L2/2` doesn't matter. This is the table that answers all three questions at once.

In [ ]:
spear_rows = []
for feed in FEEDS:
    P = STRAT[feed]; i, j, y = P['i'], P['j'], P['nl']
    for m in ACTIVE:
        s = METHODS[m]['pair'](feed, i, j)
        rho = spearmanr(s, y).correlation
        spear_rows.append(dict(method=m, feed=feed, rho=rho, n_pairs=len(y)))
spear = pd.DataFrame(spear_rows)
SPEAR_TAB = spear.pivot(index='method', columns='feed', values='rho').reindex(
    [m for m in METHODS if m in ACTIVE])[FEEDS]
print('='*60); print('SPEARMAN rho(sim, normLev)  — stratified full-range, per feed'); print('='*60)
print(SPEAR_TAB.round(3).to_string())
spear.to_csv('colab29b_spearman.csv', index=False)

## 9. Metric 2 — AUROC (full-pool exhaustive, de-hubbed — matches colab24f, NOT the Spearman sample)

**Locked design:** AUROC is the full-pool sanity check, so it streams the *entire* pairwise grid (not
the stratified Spearman pairs). For each feed we block over rows, label every unordered pair by exact
normLev, and accumulate each method's predicted-similarity histograms. **Headline = high (≥0.70) vs
random negative**; **hard-negative [0.30, 0.70) is annotated** (the harder, more honest contrast). This
is the slide-14 "is the task easy?" check — watch trigram-count/Dice collapse toward chance on SS
(3³=27) while holding up on AA/3Di.

In [ ]:
NBINS = 4000; LO, HI = -0.10, 1.10
def _binidx(p):
    return np.clip(np.floor((p - LO)/(HI - LO)*NBINS).astype(np.int64), 0, NBINS-1)
def _auroc_hist(h_pos, h_neg):
    P, Nn = h_pos.sum(), h_neg.sum()
    if P == 0 or Nn == 0: return float('nan')
    cum_below = np.cumsum(h_neg) - h_neg
    return float((h_pos*(cum_below + 0.5*h_neg)).sum()/(P*Nn))

MAX_KMER_BIN = MAX_LEN   # raw shared-3gram count is integer <= MAX_LEN-2, so integer bins are EXACT
def _score_bins(method, s):
    # 'trigram' is an unbounded integer count -> bin on its own integer scale, else it clips into the
    # last float-bin and its AUROC is meaningless. Everything else lives in ~[0,1]/cosine -> float grid.
    if method == 'trigram': return np.clip(s.astype(np.int64), 0, MAX_KMER_BIN)
    return _binidx(s)

def auroc_fullpool(feed, block=512):
    seqs = POOL_SEQ[feed]; lens = POOL_LEN[feed]; N = len(seqs)
    H = {m: {'pos': np.zeros(NBINS), 'rand': np.zeros(NBINS), 'hard': np.zeros(NBINS)} for m in ACTIVE}
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        tsim = 1.0 - Dm/den
        S = {m: METHODS[m]['block'](feed, r0, r1) for m in ACTIVE}
        for a in range(r1 - r0):
            i = r0 + a
            if i + 1 >= N: continue
            tj = tsim[a, i+1:]                       # each unordered pair counted once (j > i)
            pos = tj >= BAND_HIGH; neg = ~pos; hard = (tj >= BAND_MID) & (tj < BAND_HIGH)
            for m in ACTIVE:
                sj = S[m][a, i+1:]
                if neg.any():  H[m]['rand'] += np.bincount(_score_bins(m, sj[neg]),  minlength=NBINS)
                if pos.any():  H[m]['pos']  += np.bincount(_score_bins(m, sj[pos]),  minlength=NBINS)
                if hard.any(): H[m]['hard'] += np.bincount(_score_bins(m, sj[hard]), minlength=NBINS)
    npos = int(H[ACTIVE[0]]['pos'].sum())
    return [dict(method=m, feed=feed, n_pos=npos,
                 auroc_random=_auroc_hist(H[m]['pos'], H[m]['rand']),
                 auroc_hard=_auroc_hist(H[m]['pos'], H[m]['hard'])) for m in ACTIVE]

print('Streaming full-pool AUROC (exhaustive)...')
auroc = pd.DataFrame([r for feed in FEEDS for r in auroc_fullpool(feed)])
AUROC_TAB = auroc.pivot(index='method', columns='feed', values='auroc_random').reindex(
    [m for m in METHODS if m in ACTIVE])[FEEDS]
print('='*64); print('AUROC (full-pool) — high (>=0.70) vs RANDOM negative [headline]'); print('='*64)
print(AUROC_TAB.round(3).to_string())
HARD_TAB = auroc.pivot(index='method', columns='feed', values='auroc_hard').reindex(
    [m for m in METHODS if m in ACTIVE])[FEEDS]
print('\nAUROC vs HARD negative [0.30,0.70) [annotation]:'); print(HARD_TAB.round(3).to_string())
print('\nn_pos (high pairs) per feed:', {f: int(auroc[auroc.feed==f]['n_pos'].iloc[0]) for f in FEEDS})
auroc.to_csv('colab29b_auroc.csv', index=False)

## 10. Metric 3 — retrieval MAP@10 / hit@10 (full-pool, de-hubbed, bars 0.70 & 0.90)

Per query, rank the whole pool by each method's `query()` score; score AP@10 and hit@10 against the
exhaustive Levenshtein neighbour set. Bootstrap CI over queries. `length` is included as the floor.

In [ ]:
def _ap(order, ts, k=10):
    nt = len(ts)
    if nt == 0: return np.nan
    hits = 0; ap = 0.0
    for r, o in enumerate(order[:k], 1):
        if o in ts: hits += 1; ap += hits/r
    return ap/min(nt, k)

def retrieval(feed, method, bar):
    T = ORACLE[feed]['T_high'] if bar == 0.70 else ORACLE[feed]['T_strict']
    q_ids = list(T.keys())
    if not q_ids: return dict(bar=bar, feed=feed, method=method, n_q=0, med_T=0,
                              map=np.nan, map_lo=np.nan, map_hi=np.nan, hit=np.nan)
    qfn = METHODS[method]['query']; aps, hits = [], []
    for qi in q_ids:
        ts = set(T[qi].tolist()); order = np.argsort(-qfn(feed, qi))
        aps.append(_ap(order, ts, 10)); hits.append(1.0 if len(set(order[:10].tolist()) & ts) else 0.0)
    aps = np.array(aps)
    boot = np.array([np.nanmean(aps[brng.integers(0, len(aps), len(aps))]) for _ in range(N_BOOT)])
    return dict(bar=bar, feed=feed, method=method, n_q=len(q_ids),
                med_T=int(np.median([len(v) for v in T.values()])),
                map=float(np.nanmean(aps)), map_lo=float(np.percentile(boot, 2.5)),
                map_hi=float(np.percentile(boot, 97.5)), hit=float(np.mean(hits)))

brng = np.random.default_rng(BOOT_SEED)   # reseed each full eval -> CIs independent of cell-execution order
retr_rows = [retrieval(feed, m, bar) for bar in (0.70, 0.90) for feed in FEEDS for m in ACTIVE]
retr = pd.DataFrame(retr_rows)
retr.to_csv('colab29b_retrieval.csv', index=False)
ORD = [m for m in METHODS if m in ACTIVE]

# --- AA retrieval = hit@10 (pair-like: median|T|=1, ONE true partner per query). Bar 0.70 only;
#     AA has no >=0.90 pairs. This is the AA *control* metric (see §10c/next cell for SS/3Di MAP@10). ---
aa = retr[(retr.feed == 'AA') & (retr.bar == 0.70)].set_index('method').reindex(ORD)
print('=' * 60)
print(f'AA RETRIEVAL — hit@10 @ 0.70   (pair-like: {int(aa["n_q"].iloc[0])} queries, median|T|={int(aa["med_T"].iloc[0])})')
print('=' * 60)
print(aa[['hit']].rename(columns={'hit': 'hit@10'}).round(3).to_string())
print('\nAA is the easy control: trigram / Dice / ESM2 / SNN all saturate; only the length floor lags.')


In [ ]:
# --- SS/3Di retrieval = MAP@10 (neighbourhood: many valid exact-Lev neighbours, so hit@10 is too
#     forgiving here). Bars 0.70 (operational high-sim) and 0.90 (near-identical). MAP shown with its
#     95% bootstrap CI = the receipt for the "non-overlapping CIs" claim on the retrieval slide.
#     Set-based exact-Levenshtein oracle — say "set-based" whenever claiming the SS/3Di win. ---
for bar in (0.70, 0.90):
    sub = retr[(retr.bar == bar) & (retr.feed.isin(['SS', '3Di']))]
    cell = sub.assign(c=sub.apply(lambda r: f"{r['map']:.3f} [{r['map_lo']:.3f},{r['map_hi']:.3f}]", axis=1))
    tab = cell.pivot(index='method', columns='feed', values='c').reindex(ORD)[['SS', '3Di']]
    medT = {f: int(sub[sub.feed == f]['med_T'].iloc[0]) for f in ['SS', '3Di']}
    print('=' * 64)
    print(f'SS/3Di MAP@10 @ bar {bar:.2f}   (median|T|:  SS={medT["SS"]},  3Di={medT["3Di"]})')
    print('=' * 64)
    print(tab.to_string()); print()
print('Headline: SNN > ESM2 on SS/3Di at both bars, CIs non-overlapping. trigram/Dice/length near-zero.')


## 8b. Synthetic in-distribution column — adds a `synth` column to the Spearman & AUROC tables

The synthetic column is a **held-out** pair set from the *same generator* as the training data (uniform-AA,
seed+perturbation), drawn with an independent seed so it is **disjoint from the training pairs** — an
in-distribution generalization check, **not** the training set. It spans the full `[floor, 1.0]` normLev
range.

**Metric framing (say this on the slide):** synthetic Spearman is pair-based, exactly like the CATH Spearman
sets, so it is directly comparable. Synthetic **AUROC is computed on the held-out constructed pairs**
(pos = normLev >= 0.70 vs the rest), whereas CATH AUROC is **full-pool exhaustive** — different sampling,
same question (pairwise discrimination). Synthetic is therefore an **in-distribution reference reported via
pairwise metrics**. A synthetic *retrieval / MAP@10* score is **not** reported here — not because MAP is
undefined (its neighbourhoods would be singletons, exactly like CATH-AA, |T|=1), but because an informative
synthetic retrieval benchmark needs a deliberately constructed neighbourhood pool (one seed -> many
variants), which this pair set is not.

In [ ]:
# ---- Synthetic held-out evaluation -> adds an in-distribution 'synth' column to Spearman/AUROC ----
# The synthetic column is pair-based: seed+partner examples from the same generator family as training.
# It is intentionally NOT used for retrieval, because these pairs do not form an all-vs-all neighbourhood.

SYNTH_FEED = 'synth'
METHOD_ORDER = [m for m in ['SNN', 'ESM2', 'ProtTrans', 'Dice', 'trigram', 'length'] if m in ACTIVE]
METRIC_FEEDS = ['synth', '3Di', 'SS', 'AA']
FEED_LABEL = {'synth': 'AA synth', '3Di': '3Di', 'SS': 'SS', 'AA': 'AA CATH_s20'}

def _perturb_local(seq, k, abc, r):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = r.choice(['sub', 'del'])
        else: op = r.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = r.integers(0, len(s)); s[i] = r.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = r.integers(0, len(s) + 1); s.insert(i, r.choice(abc))
        else:
            i = r.integers(0, len(s)); del s[i]
    return ''.join(s)

def _rand_seq_local(abc, r):
    L = int(r.integers(MIN_LEN, MAX_LEN + 1))
    return ''.join(r.choice(list(abc), size=L))

def build_synth_pair_feed(n_perturb=30000, n_indep=10000, per_bin=N_STRAT_PER_BIN, seed=20260715):
    r = np.random.default_rng(seed)
    records = []

    # Perturbation pairs cover the trained operation from near-identical down toward the chance floor.
    for _ in range(n_perturb):
        base = _rand_seq_local(AA_ALPHABET, r)
        k = int(r.integers(0, len(base) + 1))
        part = _perturb_local(base, k, AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN:
            records.append((base, part, norm_lev(base, part)))

    # Independent random pairs thicken the low-similarity/chance-floor regime.
    for _ in range(n_indep):
        a = _rand_seq_local(AA_ALPHABET, r)
        b = _rand_seq_local(AA_ALPHABET, r)
        records.append((a, b, norm_lev(a, b)))

    nl_all = np.array([x[2] for x in records])
    edges = np.linspace(0.0, 1.0, 11)
    bins_all = np.clip(np.digitize(nl_all, edges) - 1, 0, 9)
    take = []
    for bb in range(10):
        idx = np.where(bins_all == bb)[0]
        if idx.size:
            take.extend(r.permutation(idx)[:per_bin].tolist())
    take = np.array(take, dtype=np.int64)

    seqs, I, J, NL, BINS = [], [], [], [], []
    for idx in take:
        a, b, nl = records[int(idx)]
        i = len(seqs); seqs.append(a)
        j = len(seqs); seqs.append(b)
        I.append(i); J.append(j); NL.append(nl); BINS.append(int(bins_all[idx]))

    return seqs, np.array(I, dtype=np.int64), np.array(J, dtype=np.int64), np.array(NL), np.array(BINS)

_ss, _I, _J, _NL, _BINS = build_synth_pair_feed()
POOL_SEQ[SYNTH_FEED] = _ss
POOL_IDS[SYNTH_FEED] = [f'syn{i}' for i in range(len(_ss))]
POOL_LEN[SYNTH_FEED] = np.array([len(s) for s in _ss])
LOOK[SYNTH_FEED] = {f'syn{i}': s for i, s in enumerate(_ss)}
SNN_BY_FEED[SYNTH_FEED] = model_aa                      # D1: same frozen AA encoder
STRAT[SYNTH_FEED] = dict(i=_I, j=_J, nl=_NL, bins=_BINS)

_pos = _NL >= BAND_HIGH
_hrd = (_NL >= BAND_MID) & (_NL < BAND_HIGH)
print(f'synth pair set: {len(_NL)} pairs, {len(_ss)} sequences | '
      f'pos(>=0.70)={int(_pos.sum())}  hard[0.30,0.70)={int(_hrd.sum())}')
print('synth count per normLev decile:', [int((_BINS == b).sum()) for b in range(10)])

synth_rows_spear, synth_rows_auroc = [], []
synth_pair_eval = pd.DataFrame({'feed': SYNTH_FEED, 'i': _I, 'j': _J, 'normLev': _NL, 'bin': _BINS})
for m in ACTIVE:
    s = METHODS[m]['pair'](SYNTH_FEED, _I, _J)
    synth_pair_eval[m] = s
    rho = spearmanr(s, _NL).correlation
    au_rand = roc_auc_score(_pos.astype(int), s) if _pos.any() and (~_pos).any() else np.nan
    msk = _pos | _hrd
    au_hard = roc_auc_score(_pos[msk].astype(int), s[msk]) if _pos.any() and _hrd.any() else np.nan
    synth_rows_spear.append(dict(method=m, feed=SYNTH_FEED, rho=rho, n_pairs=len(_NL)))
    synth_rows_auroc.append(dict(method=m, feed=SYNTH_FEED, n_pos=int(_pos.sum()),
                                 auroc_random=au_rand, auroc_hard=au_hard,
                                 sampling='pairwise_synthetic'))

# Rebuild the metric tables with synth included, avoiding duplicate synth rows if this cell is re-run.
spear = pd.concat([spear[spear.feed != SYNTH_FEED], pd.DataFrame(synth_rows_spear)], ignore_index=True)
auroc = pd.concat([auroc[auroc.feed != SYNTH_FEED], pd.DataFrame(synth_rows_auroc)], ignore_index=True)
TABLE_FEEDS = [f for f in METRIC_FEEDS if f in set(spear.feed)]
SPEAR_TAB = spear.pivot(index='method', columns='feed', values='rho').reindex(index=METHOD_ORDER, columns=TABLE_FEEDS)
AUROC_TAB = auroc.pivot(index='method', columns='feed', values='auroc_random').reindex(index=METHOD_ORDER, columns=TABLE_FEEDS)
HARD_TAB = auroc.pivot(index='method', columns='feed', values='auroc_hard').reindex(index=METHOD_ORDER, columns=TABLE_FEEDS)

spear.to_csv('colab29b_spearman.csv', index=False)
auroc.to_csv('colab29b_auroc.csv', index=False)
synth_pair_eval.round(4).to_csv('colab29b_synth_pair_eval.csv', index=False)

print('\nSpearman with synthetic in-distribution column:')
print(SPEAR_TAB.round(3).to_string())
print('\nAUROC with synthetic in-distribution column:')
print(AUROC_TAB.round(3).to_string())
print('\nNOTE: synth AUROC is pair-wise on constructed pairs; CATH AUROC is full-pool exhaustive.')


## 10b. All numbers in one place — Spearman + AUROC + MAP@10 + hit@10

Every metric for every method × feed in a single table (and CSV), so the slide numbers don't have to be
read off the bar charts. Reminder on the metric split: **read AA via hit@10** (pair-like), **read SS/3Di
via MAP@10** (neighbourhood). AA has no ≥0.90 pairs, so its 0.90 columns are NaN by construction.

In [ ]:
def _get(df, m, f, col, **filt):
    q = (df.method == m) & (df.feed == f)
    for k, v in filt.items():
        q &= (df[k] == v)
    s = df[q][col]
    return float(s.values[0]) if len(s) else np.nan

SUMMARY_FEEDS = [f for f in globals().get('METRIC_FEEDS', FEEDS) if (f in set(spear.feed) or f in FEEDS)]
SUMMARY_METHODS = [m for m in globals().get('METHOD_ORDER', [m for m in METHODS if m in ACTIVE]) if m in ACTIVE]

summary = pd.DataFrame([dict(
    feed=f, method=m,
    metric_sampling='pair-wise synthetic' if f == 'synth' else 'full-pool CATH',
    spearman=_get(spear, m, f, 'rho'),
    AUROC_rand=_get(auroc, m, f, 'auroc_random'),
    AUROC_hard=_get(auroc, m, f, 'auroc_hard'),
    **{'MAP@0.70': _get(retr, m, f, 'map', bar=0.70), 'MAP@0.70_lo': _get(retr, m, f, 'map_lo', bar=0.70),
       'MAP@0.70_hi': _get(retr, m, f, 'map_hi', bar=0.70), 'hit@0.70': _get(retr, m, f, 'hit', bar=0.70),
       'MAP@0.90': _get(retr, m, f, 'map', bar=0.90), 'hit@0.90': _get(retr, m, f, 'hit', bar=0.90)})
    for f in SUMMARY_FEEDS for m in SUMMARY_METHODS])

pd.set_option('display.width', 240); pd.set_option('display.max_columns', 30)
print('=' * 118)
print('CONSOLIDATED METRICS — Spearman | AUROC | retrieval')
print('synth: pair-wise geometry/discrimination only  |  CATH: full-pool AUROC + retrieval')
print('read AA CATH via hit@10 (pair-like)  |  read SS/3Di via MAP@10 (neighbourhood)')
print('=' * 118)
print(summary.round(3).to_string(index=False))
summary.round(4).to_csv('colab29b_all_metrics.csv', index=False)
print('\nSaved colab29b_all_metrics.csv')


## 11. Figures — despined, presentation style

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
plt.rcParams.update({'font.size': 11})
def despine(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Presentation palette: SNN = strong blue, ESM2 = pale green, length = grey,
# trigram/Dice = related orange shades because both are k-mer baselines.
COLORS = {
    'SNN':       '#0072B2',
    'ESM2':      '#8FD694',
    'Dice':      '#E69F00',
    'trigram':   '#F4A261',
    'length':    '#8E99A5',
    'ProtTrans': '#C8C8FF',
}

METHOD_ORDER = [m for m in globals().get('METHOD_ORDER', ['SNN', 'ESM2', 'ProtTrans', 'Dice', 'trigram', 'length']) if m in ACTIVE]
METRIC_FEEDS = [f for f in globals().get('METRIC_FEEDS', ['synth', '3Di', 'SS', 'AA'])]
FEED_LABEL = globals().get('FEED_LABEL', {'synth': 'AA synth', '3Di': '3Di', 'SS': 'SS', 'AA': 'AA CATH_s20'})
COL_LABEL = {'synth': 'AA\nsynth', '3Di': '3Di', 'SS': 'SS', 'AA': 'AA\nCATH_s20'}

def heatmap(TAB, title, fname, center, vspan, txt_gap, cbar_label, note=None):
    cols = [c for c in METRIC_FEEDS if c in TAB.columns]
    rows = [m for m in METHOD_ORDER if m in TAB.index]
    M = TAB.reindex(index=rows, columns=cols).values.astype(float)
    norm = TwoSlopeNorm(vmin=center - vspan, vcenter=center, vmax=center + vspan)
    fig, ax = plt.subplots(figsize=(1.18 * len(cols) + 1.9, 0.74 * len(rows) + 1.45))
    im = ax.imshow(M, cmap='RdBu_r', norm=norm, aspect='auto')   # low = blue, high = red
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([COL_LABEL.get(c, c) for c in cols])
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows)
    for r in range(M.shape[0]):
        for c in range(M.shape[1]):
            v = M[r, c]
            ax.text(c, r, '—' if np.isnan(v) else f'{v:.2f}', ha='center', va='center', fontsize=12,
                    color='white' if (not np.isnan(v) and abs(v - center) > txt_gap) else 'black')
    ax.set_title(title, fontsize=12)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(cbar_label)
    if note:
        fig.text(0.5, -0.02, note, ha='center', fontsize=8, color='dimgray')
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    ordered = TAB.reindex(index=rows, columns=cols)
    ordered.to_csv(fname.replace('.png', '.csv'))
    plt.show()

# --- Centerpiece 1: Spearman — white at 0; negative cells render blue, high positive cells red ---
heatmap(SPEAR_TAB, 'Spearman rho(sim, normLev) - stratified pair sets',
        'colab29b_spearman.png', center=0.0, vspan=1.0, txt_gap=0.55, cbar_label='rho',
        note='AA synth = in-distribution reference; AA CATH_s20 = redundancy-reduced low-similarity control.')


### AUROC heatmap (centerpiece companion to the Spearman table)

Same diverging style, but white is centered at **chance (0.5)**, not 0 — so below-chance cells (trigram
SS/3Di) render **blue** = the k-mer-count collapse; the SNN row is uniformly deep red. The synthetic column
is AUROC on **held-out constructed pairs** (pairwise discrimination), whereas the CATH columns are
**full-pool exhaustive** — same question, different sampling (see 8b).

In [ ]:
# --- Centerpiece 2: AUROC heatmaps — diverging, white at CHANCE (0.5); below-chance renders blue ---
heatmap(AUROC_TAB, 'AUROC (high >= 0.70 vs background)',
        'colab29b_auroc_heatmap.png', center=0.5, vspan=0.5, txt_gap=0.28, cbar_label='AUROC',
        note='AA synth AUROC is pair-wise on constructed pairs; 3Di/SS/AA CATH_s20 AUROC is full-pool exhaustive.')

heatmap(HARD_TAB, 'AUROC (high >= 0.70 vs hard negatives [0.30, 0.70))',
        'colab29b_auroc_hard_heatmap.png', center=0.5, vspan=0.5, txt_gap=0.28, cbar_label='AUROC',
        note='Hard-negative AUROC asks whether methods separate high-similarity pairs from genuinely similar-ish non-hits.')


### Diagnostic scatter matrix — method scores against each other

This is a **sanity/check-for-contention** figure, not a headline metric. It samples the same pair sets used
for Spearman (including AA synth) and plots all pairwise combinations of `normLev`, SNN, ESM2, Dice,
trigram-count, and length-only. Use it to spot hidden confounds, e.g. SNN collapsing to length, Dice/trigram
tracking each other, or one alphabet driving a claimed relationship.


In [ ]:
# --- Diagnostic pairwise scatter over all feeds/method scores ---
SCATTER_METHODS = [m for m in ['SNN', 'ESM2', 'ProtTrans', 'Dice', 'trigram', 'length'] if m in ACTIVE]
SCATTER_FEEDS = [f for f in METRIC_FEEDS if f in STRAT]
SCATTER_N_PER_FEED = 1200
SCATTER_COLORS = {'synth': '#222222', '3Di': '#D55E00', 'SS': '#009E73', 'AA': '#0072B2'}

def pair_score_frame(feed, n=SCATTER_N_PER_FEED, seed=17):
    r = np.random.default_rng(seed + sum(ord(ch) for ch in feed))
    P = STRAT[feed]
    idx = np.arange(len(P['nl']))
    if len(idx) > n:
        idx = r.choice(idx, n, replace=False)
    ii, jj, nl = P['i'][idx], P['j'][idx], P['nl'][idx]
    out = pd.DataFrame({'feed': feed, 'normLev': nl})
    for m in SCATTER_METHODS:
        out[m] = METHODS[m]['pair'](feed, ii, jj)
    return out

pair_scores = pd.concat([pair_score_frame(f) for f in SCATTER_FEEDS], ignore_index=True)
pair_scores.round(4).to_csv('colab29b_method_pair_scores_sample.csv', index=False)

PAIR_VARS = ['normLev'] + SCATTER_METHODS
n = len(PAIR_VARS)
fig, axes = plt.subplots(n, n, figsize=(2.15 * n, 2.15 * n))

for i, yvar in enumerate(PAIR_VARS):
    for j, xvar in enumerate(PAIR_VARS):
        ax = axes[i, j]
        if i == j:
            for feed, grp in pair_scores.groupby('feed'):
                ax.hist(grp[xvar].dropna(), bins=28, alpha=0.35, density=True,
                        color=SCATTER_COLORS.get(feed, 'grey'), label=FEED_LABEL.get(feed, feed))
            ax.set_yticks([])
        elif i > j:
            for feed, grp in pair_scores.groupby('feed'):
                ax.scatter(grp[xvar], grp[yvar], s=7, alpha=0.22,
                           color=SCATTER_COLORS.get(feed, 'grey'), linewidths=0)
        else:
            ax.axis('off')
            y0 = 0.88
            for feed, grp in pair_scores.groupby('feed'):
                ok = np.isfinite(grp[xvar]) & np.isfinite(grp[yvar])
                if ok.sum() > 2 and grp.loc[ok, xvar].nunique() > 1 and grp.loc[ok, yvar].nunique() > 1:
                    rho = spearmanr(grp.loc[ok, xvar], grp.loc[ok, yvar]).correlation
                    txt = f'{FEED_LABEL.get(feed, feed)}: {rho:.2f}'
                else:
                    txt = f'{FEED_LABEL.get(feed, feed)}: —'
                ax.text(0.05, y0, txt, transform=ax.transAxes, fontsize=8,
                        color=SCATTER_COLORS.get(feed, 'grey'))
                y0 -= 0.17
        if i == n - 1:
            ax.set_xlabel(xvar, fontsize=9)
        else:
            ax.set_xticklabels([])
        if j == 0 and i != 0:
            ax.set_ylabel(yvar, fontsize=9)
        elif j != 0:
            ax.set_yticklabels([])
        if i >= j:
            despine(ax)

handles = [plt.Line2D([0], [0], marker='o', color='w', label=FEED_LABEL.get(f, f),
                      markerfacecolor=SCATTER_COLORS.get(f, 'grey'), markersize=7)
           for f in SCATTER_FEEDS]
fig.legend(handles=handles, loc='upper center', ncol=len(handles), frameon=False)
fig.suptitle('Diagnostic scatter matrix: method scores on the Spearman pair sets', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('colab29b_method_pair_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('Diagnostic guard: this scatter matrix samples the Spearman pair sets, not the full CATH pool.')
print('Use it to find confounds/contention points; do not present it as a fourth evaluation metric.')


In [ ]:
# --- Slide 15: separation view — SNN predicted similarity, high (>=0.70) vs low/mid (<0.70) pairs ---
# Box + jittered strip per feed (AA / SS / 3Di), with the full-pool AUROC (Section 9) annotated.
# Boxes are intentionally very transparent so the individual points remain visible.
ENC_LABEL = {'AA': 'AA-enc', 'SS': 'AA-enc', '3Di': 'AA-enc'}   # D1: one AA-trained encoder everywhere

def separation_panel(ax, feed, n_pos_cap=20000, n_neg=40000, strip_cap=2000, seed=0):
    E = get_emb('snn', feed)
    N = E.shape[0]
    r = np.random.default_rng(seed)
    seqs = POOL_SEQ[feed]

    # high (>=0.70) positives from the exhaustive oracle (sampled if huge, e.g. SS)
    pp = ORACLE[feed]['pos_pairs']
    n_pos_true = len(pp)
    if pp:
        pa = np.array(pp, dtype=np.int64)
        if len(pa) > n_pos_cap:
            pa = pa[r.choice(len(pa), n_pos_cap, replace=False)]
        hi = 1.0 - np.linalg.norm(E[pa[:, 0]] - E[pa[:, 1]], axis=1) / 2.0
    else:
        hi = np.array([])

    # low/mid (<0.70) negatives: random pool pairs, labelled by exact normLev
    a = r.integers(0, N, n_neg)
    b = r.integers(0, N, n_neg)
    ok = a != b
    a, b = a[ok], b[ok]

    nl = np.array([norm_lev(seqs[x], seqs[y]) for x, y in zip(a, b)])
    lm = nl < BAND_HIGH
    a, b = a[lm], b[lm]
    lo = 1.0 - np.linalg.norm(E[a] - E[b], axis=1) / 2.0

    # Youden threshold on the shown sample (illustrative)
    thr = np.nan
    if len(hi):
        grid = np.linspace(min(lo.min(), hi.min()), max(lo.max(), hi.max()), 400)
        J = [(hi >= t).mean() - (lo >= t).mean() for t in grid]
        thr = float(grid[int(np.argmax(J))])

    # Very transparent boxplot behind the points
    bp = ax.boxplot(
        [lo, hi],
        positions=[0, 1],
        widths=0.5,
        showfliers=False,
        patch_artist=True,
        zorder=1
    )

    for patch, col in zip(bp['boxes'], ['#d9d9d9', '#f2c2c2']):
        patch.set_facecolor(col)
        patch.set_alpha(0.12)
        patch.set_edgecolor(col)
        patch.set_linewidth(1.0)
        patch.set_zorder(1)

    for whisker in bp['whiskers']:
        whisker.set_color('#777777')
        whisker.set_alpha(0.35)
        whisker.set_zorder(1)

    for cap in bp['caps']:
        cap.set_color('#777777')
        cap.set_alpha(0.35)
        cap.set_zorder(1)

    for med in bp['medians']:
        med.set_color('black')
        med.set_alpha(0.85)
        med.set_linewidth(1.5)
        med.set_zorder(4)

    # Jittered points on top of the transparent boxes
    for grp, x0, col in [(lo, 0, '#888888'), (hi, 1, '#c0392b')]:
        if not len(grp):
            continue

        s = grp if len(grp) <= strip_cap else r.choice(grp, strip_cap, replace=False)

        ax.scatter(
            x0 + r.uniform(-0.17, 0.17, len(s)),
            s,
            s=13 if feed == 'AA' else 7,
            alpha=0.85 if feed == 'AA' else 0.38,
            color=col,
            linewidths=0,
            zorder=3
        )

    if not np.isnan(thr):
        ax.axhline(thr, ls='--', color='black', lw=1, zorder=2)
        ax.text(1.52, thr, ' separation\n threshold', va='center', fontsize=8)

    ax.set_xlim(-0.6, 1.95)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([
        f'low/mid\n(<0.70)\nn={len(lo):,}',
        f'high\n(>=0.70)\nn={n_pos_true:,}'
    ])
    ax.set_title(
        f'{feed} ({ENC_LABEL[feed]})   AUROC = {float(AUROC_TAB.loc["SNN", feed]):.3f}',
        fontsize=11
    )
    despine(ax)


fig, axes = plt.subplots(1, 3, figsize=(13, 4.8), sharey=True)

for ax, feed in zip(axes, FEEDS):
    separation_panel(ax, feed)

axes[0].set_ylabel('SNN predicted similarity\n1 - ||e_a - e_b|| / 2')

plt.suptitle(
    'Does the SNN geometry separate high-sim pairs from the rest? (per alphabet)',
    y=1.02,
    fontsize=12
)

fig.text(
    0.5,
    -0.05,
    'Transparent box + strip: random sample of the exhaustive pair sets '
    '(AA high = all 5 real pairs). AUROC is full-pool (Section 9); '
    'the dashed line is an illustrative Youden threshold.',
    ha='center',
    fontsize=8,
    color='dimgray'
)

plt.tight_layout()
plt.savefig('colab29b_separation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Slide 16: retrieval "up close" — query cards. For a query, show the SNN's top-k neighbours with
#     true exact-Lev partners (>=0.70) starred + highlighted, and the query's own hit@10 / MAP@10.
#     Metric split is baked in: AA cards read hit@10 (one true partner), SS/3Di read MAP@10 (a set). ---
def query_ranking(feed, qid, method='SNN', k=12):
    ids = POOL_IDS[feed]
    if qid not in LOOK[feed]:
        print(f'  [skip] {qid} not in {feed} pool (failed that feed\'s validity filter)'); return None, None
    qi = ids.index(qid); look = LOOK[feed]
    sim = METHODS[method]['query'](feed, qi)                     # sim vs whole pool, self = -inf
    ts = set(ORACLE[feed]['T_high'].get(qi, np.empty(0, np.int32)).tolist())   # true neighbours >=0.70
    full = np.argsort(-sim)
    ap = _ap(full, ts, 10); hit = 1.0 if len(set(full[:10].tolist()) & ts) else 0.0
    rows = [dict(rank=r, domain=ids[o], sim=float(sim[o]),
                 normLev=norm_lev(look[qid], look[ids[o]]), true=(o in ts))
            for r, o in enumerate(full[:k], 1)]
    return pd.DataFrame(rows), dict(feed=feed, qid=qid, method=method, nT=len(ts), ap=ap, hit=hit)

def render_card(ax, feed, qid, method='SNN', k=10):
    df, meta = query_ranking(feed, qid, method, k); ax.axis('off')
    if df is None:
        ax.set_title(f'{feed}: {qid}\n(not in pool)', fontsize=9); return
    cells = [[r['rank'], r['domain'], f"{r['sim']:.3f}", f"{r['normLev']:.2f}", '★' if r['true'] else '']
             for _, r in df.iterrows()]
    tbl = ax.table(cellText=cells, colLabels=['#', 'neighbour', 'SNN sim', 'normLev', 'match'],
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.3)
    for (rr, cc), cell in tbl.get_celld().items():
        if rr == 0: cell.set_text_props(weight='bold')
        elif df.iloc[rr - 1]['true']: cell.set_facecolor('#f6d5d5')
    metric = f"hit@10 = {meta['hit']:.0f}" if feed == 'AA' else f"MAP@10 = {meta['ap']:.2f}"
    ax.set_title(f"{feed}: query {qid}\n|T| (true ≥0.70 neighbours) = {meta['nT']}    {metric}", fontsize=9)

# Candidate queries per feed — pick SS/3Di cards with a small-but-nontrivial true set (|T|≈2..8) so a
# partly-filled top-10 (MAP@10 < 1) is legible. Prints (pool_idx, |T|, domain_id).
print('Candidate queries per feed (idx, |T|, domain) — a good MAP demo has |T| ~ 2..8:')
for f in FEEDS:
    cand = sorted(ORACLE[f]['T_high'].items(), key=lambda kv: len(kv[1]))
    picks = [c for c in cand if 2 <= len(c[1]) <= 8][:6] or cand[:6]
    print(f'  {f:<4}', [(int(i), int(len(v)), POOL_IDS[f][i]) for i, v in picks])

# EDIT this list after picking SS/3Di examples from the candidates above (AA stays pair-like).
CARD_QUERIES = [('AA', '1toIA01'), ('AA', '2k3oA00')]   # e.g. + [('SS', '<id>'), ('3Di', '<id>')]
fig, axes = plt.subplots(1, len(CARD_QUERIES), figsize=(4.3 * len(CARD_QUERIES), 4.2))
axes = np.atleast_1d(axes)
for ax, (f, q) in zip(axes, CARD_QUERIES): render_card(ax, f, q)
plt.tight_layout(); plt.savefig('colab29b_query_cards.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# --- AUROC grouped bars (same method/feed order as the heatmap, values on bars) ---
plot_feeds = [f for f in METRIC_FEEDS if f in AUROC_TAB.columns]
plot_methods = [m for m in METHOD_ORDER if m in AUROC_TAB.index]
x = np.arange(len(plot_feeds))
w = 0.8 / len(plot_methods)

fig, ax = plt.subplots(figsize=(10, 5.2))
for k, m in enumerate(plot_methods):
    vals = [float(AUROC_TAB.loc[m, f]) for f in plot_feeds]
    xoff = x + (k - (len(plot_methods) - 1) / 2) * w
    bars = ax.bar(xoff, vals, w, label=m, color=COLORS.get(m, None))
    ax.bar_label(bars, labels=[f'{v:.2f}' if np.isfinite(v) else '—' for v in vals], padding=2, fontsize=8)

ax.axhline(0.5, ls='--', color='grey', lw=1)
ax.set_ylim(0, 1.16)
ax.set_xticks(x)
ax.set_xticklabels([COL_LABEL.get(f, f) for f in plot_feeds])
ax.set_ylabel('AUROC (high >= 0.70 vs background)')
ax.set_title('Pairwise discrimination — ordered as in the Spearman table')
ax.text(0.5, -0.20, 'AA synth is pair-wise on constructed pairs; CATH feeds are full-pool exhaustive.',
        transform=ax.transAxes, ha='center', fontsize=8, color='dimgray')
ax.legend(ncol=len(plot_methods), fontsize=9, loc='lower center')
despine(ax)

plt.tight_layout()
plt.savefig('colab29b_auroc.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- MAP@10 grouped bars, value labels on top (CI whiskers dropped on request) ---
# NOTE: the 95% bootstrap CIs are NOT gone — they are in `retr` / colab29b_retrieval.csv and printed in
# §10. The slide-17 "non-overlapping CIs" claim still holds; put the intervals in the slide caption.
def map_chart(bar):
    sub = retr[retr.bar == bar]; x = np.arange(len(FEEDS)); w = 0.8/len(ACTIVE)
    fig, ax = plt.subplots(figsize=(9, 5))
    for k, m in enumerate(ACTIVE):
        p = [float(sub[(sub.method==m)&(sub.feed==f)]['map'].values[0]) for f in FEEDS]
        xoff = x + (k-(len(ACTIVE)-1)/2)*w
        b = ax.bar(xoff, p, w, label=m, color=COLORS.get(m, None))
        ax.bar_label(b, fmt='%.2f', fontsize=8, padding=2)
    ax.set_ylim(0, 1.12); ax.set_xticks(x); ax.set_xticklabels(FEEDS)
    ax.set_ylabel('MAP@10 (vs exact-Lev neighbour set)')
    ax.set_title(f'Full-pool retrieval MAP@10 — bar {bar:.2f}' +
                 ('  (operational high-sim)' if bar == 0.70 else '  (well-posed near-identical)'))
    ax.legend(ncol=len(ACTIVE), fontsize=9, loc='upper right'); despine(ax)
    plt.tight_layout(); plt.savefig(f'colab29b_map_{int(bar*100)}.png', dpi=150, bbox_inches='tight'); plt.show()
map_chart(0.70); map_chart(0.90)


### Retrieval close-up — SNN vs ESM2 vs length (hit@10 @ 0.70)

D1 point estimates (these match `retr` / `colab29_retrieval.csv`; the CATH feeds are unaffected by the
synthetic column). If a future re-run changes the numbers, refresh the values in the frame below.

In [ ]:
# --- Retrieval comparison: SNN vs ESM2 vs ProtTrans vs length (hit@10 @ 0.70) ---
# LIVE from `retr` (built in §10) so ProtTrans folds in automatically once run. Uses the COLORS dict.
methods = [m for m in ['SNN', 'ESM2', 'ProtTrans', 'length'] if m in ACTIVE]
feeds = ['AA', 'SS', '3Di']
sub70 = retr[retr.bar == 0.70]

x = np.arange(len(feeds)); w = 0.8 / len(methods)
fig, ax = plt.subplots(figsize=(9.8, 5))
for k, method in enumerate(methods):
    y = [float(sub70[(sub70.method == method) & (sub70.feed == f)]['hit'].values[0]) for f in feeds]
    xoff = x + (k - (len(methods) - 1) / 2) * w
    bars = ax.bar(xoff, y, w, label=method, color=COLORS.get(method, None))
    ax.bar_label(bars, labels=[f'{v:.2f}' for v in y], padding=3, fontsize=8)

ax.axhline(0, color='black', lw=0.8)
ax.set_ylim(0, 1.15)
ax.set_xticks(x); ax.set_xticklabels(feeds)
ax.set_ylabel('Hit@10 @ 0.70')
ax.set_title('Retrieval comparison against exact-Levenshtein neighbours')
ax.legend(ncol=len(methods), fontsize=9, loc='upper right')
despine(ax)

plt.tight_layout()
plt.savefig('colab29b_retrieval_snn_esm2_prottrans_length_hit70.png', dpi=150, bbox_inches='tight')
plt.show()

### Appendix — scatter grid (one panel per method × alphabet: 5×4, or 6×4 with ProtTrans live): predicted similarity vs true normLev (the Spearman picture)

One panel per method × alphabet. Faint 2-D density (hexbin) behind, low-alpha dots on top, and a
per-decile **median trend line** so the monotonic (or negative) trend is visible. The ρ printed on each
panel is from the same stratified pair set, so it matches the Spearman heatmap cell exactly. What to look
for: SNN on SS/3Di/synthetic = tight rising band; SNN on AA-CATH = flat smear squeezed into the low-normLev
strip (no training support there, not failure); **length on AA-CATH and trigram on 3Di tilt downward**
(negative ρ). MAP has no pairwise scatter — it stays on the bars + query cards.

In [ ]:
# --- Scatter: predicted similarity (y) vs true normLev (x), per method x alphabet ---
# Saves ONE standalone image per (method, alphabet) combination (prof's request) + a combined grid overview.
def _pairs_for(feed):
    P = STRAT[feed]
    return P['i'], P['j'], P['nl']

SCAT_ROWS = [m for m in METHOD_ORDER if m in ACTIVE]                 # SNN, ESM2, Dice, trigram, length
SCAT_COLS = [c for c in METRIC_FEEDS if c in SPEAR_TAB.columns]      # synth, 3Di, SS, AA
edges = np.linspace(0, 1, 11); centers = 0.5 * (edges[:-1] + edges[1:])
_rg = np.random.default_rng(0)
_safe = lambda s: s.replace('/', '').replace(' ', '').replace('\n', '')

def draw_scatter(ax, m, f, title=None):
    i, j, yt = _pairs_for(f)
    s = np.asarray(METHODS[m]['pair'](f, i, j), dtype=float)
    yt = np.asarray(yt, dtype=float)
    ok = np.isfinite(s) & np.isfinite(yt); s, yt = s[ok], yt[ok]
    # faint density behind so the dense mass reads instead of a black blob
    if len(s) > 20:
        ax.hexbin(yt, s, gridsize=28, cmap='Greys', mincnt=1, alpha=0.35, linewidths=0, zorder=1)
    # capped, low-alpha dots on top so individual points stay visible
    sel = _rg.choice(len(s), 2000, replace=False) if len(s) > 2000 else np.arange(len(s))
    ax.scatter(yt[sel], s[sel], s=6, alpha=0.25, color=COLORS.get(m, '#444444'), linewidths=0, zorder=2)
    # per-decile median trend line (what Spearman rewards: monotonicity)
    b = np.clip(np.digitize(yt, edges) - 1, 0, 9); mx, my = [], []
    for k in range(10):
        msk = b == k
        if msk.sum() >= 5:
            mx.append(centers[k]); my.append(np.median(s[msk]))
    if mx:
        ax.plot(mx, my, '-o', color='black', lw=1.4, ms=2.5, zorder=4)
    rho = spearmanr(s, yt).correlation
    ax.text(0.04, 0.94, f'rho={rho:.2f}', transform=ax.transAxes, va='top', ha='left',
            fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.7))
    if title:
        ax.set_title(title, fontsize=11)
    ax.set_xlim(0, 1); despine(ax)

# (1) standalone images — one per (method, alphabet) combination
saved = []
for m in SCAT_ROWS:
    for f in SCAT_COLS:
        fig, ax = plt.subplots(figsize=(4.2, 3.6))
        draw_scatter(ax, m, f, title=f'{m} on {FEED_LABEL.get(f, f)}')
        ax.set_xlabel('true normLev'); ax.set_ylabel(f'{m} predicted similarity')
        plt.tight_layout()
        fname = f'colab29b_scatter_{_safe(FEED_LABEL.get(f, f))}_{m}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight'); plt.close(fig); saved.append(fname)
print(f'saved {len(saved)} standalone scatter images:'); print('  ' + '\n  '.join(saved))

# (2) combined grid overview (drop this block if you only want the standalone files)
nR, nC = len(SCAT_ROWS), len(SCAT_COLS)
fig, axes = plt.subplots(nR, nC, figsize=(3.1 * nC, 2.7 * nR), sharex=True, sharey='row')
axes = np.atleast_2d(axes)
for rr, m in enumerate(SCAT_ROWS):
    for cc, f in enumerate(SCAT_COLS):
        ax = axes[rr][cc]
        draw_scatter(ax, m, f, title=(FEED_LABEL.get(f, f) if rr == 0 else None))
        if cc == 0:      ax.set_ylabel(f'{m}\npredicted sim', fontsize=9)
        if rr == nR - 1: ax.set_xlabel('true normLev', fontsize=9)
fig.suptitle('Predicted similarity vs true normLev — method × alphabet (the Spearman picture)',
             y=1.004, fontsize=13)
plt.tight_layout()
plt.savefig('colab29b_scatter_grid.png', dpi=150, bbox_inches='tight')
plt.show()

### Appendix — AUROC separation plots (slide-15 style, one per method × alphabet: 20, or 24 with ProtTrans live)

The AUROC counterpart to the Spearman scatter grid. Each panel is the slide-15 separation view — predicted
similarity of **high (≥0.70)** pairs (red) vs **low/mid (<0.70)** pairs (grey), box + jittered strip — for
one method × alphabet, with the **reported AUROC** (from `AUROC_TAB`, full-pool for CATH / pair-wise for
synth) annotated. Saves 20 standalone images plus a combined grid. AA-CATH's high group is ~5 real pairs by
construction — shown honestly.

In [ ]:
# --- AUROC separation plots (one per method x alphabet): high (>=0.70) vs low/mid (<0.70) predicted-sim ---
# AUROC counterpart to the Spearman scatter grid; same stratified pair sets, collapsed into high vs low.
def draw_separation(ax, m, f, seed=0, strip_cap=1500, title=None):
    P = STRAT[f]
    s = np.asarray(METHODS[m]['pair'](f, P['i'], P['j']), dtype=float)
    nl = np.asarray(P['nl'], dtype=float)
    ok = np.isfinite(s) & np.isfinite(nl); s, nl = s[ok], nl[ok]
    lo = s[nl < BAND_HIGH]; hi = s[nl >= BAND_HIGH]
    r = np.random.default_rng(seed)

    # transparent boxes behind (only for non-empty groups)
    box_data, box_pos, box_face = [], [], []
    for d, pos, fc in [(lo, 0, '#d9d9d9'), (hi, 1, '#f2c2c2')]:
        if len(d): box_data.append(d); box_pos.append(pos); box_face.append(fc)
    if box_data:
        bp = ax.boxplot(box_data, positions=box_pos, widths=0.5, showfliers=False,
                        patch_artist=True, zorder=1)
        for patch, fc in zip(bp['boxes'], box_face):
            patch.set_facecolor(fc); patch.set_alpha(0.15); patch.set_edgecolor(fc)
        for med in bp['medians']: med.set_color('black'); med.set_linewidth(1.4); med.set_zorder(4)
        for wk in bp['whiskers']: wk.set_color('#888888'); wk.set_alpha(0.4)
        for cp in bp['caps']:     cp.set_color('#888888'); cp.set_alpha(0.4)

    # jittered strips on top (bigger/opaquer when the group is tiny, e.g. AA high = ~5)
    for grp, x0, col in [(lo, 0, '#888888'), (hi, 1, '#c0392b')]:
        if not len(grp): continue
        g = grp if len(grp) <= strip_cap else r.choice(grp, strip_cap, replace=False)
        small = len(grp) < 50
        ax.scatter(x0 + r.uniform(-0.17, 0.17, len(g)), g, s=13 if small else 6,
                   alpha=0.8 if small else 0.3, color=col, linewidths=0, zorder=3)

    ax.set_xlim(-0.6, 1.6); ax.set_xticks([0, 1])
    ax.set_xticklabels([f'low/mid\n(<0.70)\nn={len(lo):,}', f'high\n(>=0.70)\nn={len(hi):,}'], fontsize=8)
    au = float(AUROC_TAB.loc[m, f]) if (m in AUROC_TAB.index and f in AUROC_TAB.columns) else np.nan
    base = title or f'{m} on {FEED_LABEL.get(f, f)}'
    ax.set_title(base + (f'   AUROC={au:.3f}' if np.isfinite(au) else ''), fontsize=10)
    despine(ax)

SEP_ROWS = [m for m in METHOD_ORDER if m in ACTIVE]
SEP_COLS = [c for c in METRIC_FEEDS if c in STRAT]

# (1) standalone images — one per (method, alphabet) combination
sep_saved = []
for m in SEP_ROWS:
    for f in SEP_COLS:
        fig, ax = plt.subplots(figsize=(4.0, 3.8))
        draw_separation(ax, m, f)
        ax.set_ylabel(f'{m} predicted similarity')
        plt.tight_layout()
        fname = f'colab29b_auroc_sep_{_safe(FEED_LABEL.get(f, f))}_{m}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight'); plt.close(fig); sep_saved.append(fname)
print(f'saved {len(sep_saved)} standalone AUROC separation images:'); print('  ' + '\n  '.join(sep_saved))

# (2) combined grid overview (drop this block if you only want the standalone files)
nR, nC = len(SEP_ROWS), len(SEP_COLS)
fig, axes = plt.subplots(nR, nC, figsize=(3.2 * nC, 3.0 * nR), sharey='row')
axes = np.atleast_2d(axes)
for rr, m in enumerate(SEP_ROWS):
    for cc, f in enumerate(SEP_COLS):
        draw_separation(axes[rr][cc], m, f, title=(FEED_LABEL.get(f, f) if rr == 0 else f'{m} · {FEED_LABEL.get(f, f)}'))
        if cc == 0: axes[rr][cc].set_ylabel(f'{m}\npredicted sim', fontsize=9)
fig.suptitle('AUROC separation — high (≥0.70) vs low/mid, method × alphabet', y=1.003, fontsize=13)
plt.tight_layout()
plt.savefig('colab29b_auroc_sep_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 11b. Exhaustive natural-CATH normLev distribution — *why the AA data is hard*

Streams **every unordered pair** of a feed's pool (no full matrix stored) and bins normLev. This proves
the [0.4–0.7] gap in AA is real, not a sampling artefact: CATH-S20 is redundancy-reduced, so natural AA
is a mass of low-similarity pairs, ~5 high-sim pairs, and an **empty middle** — which is exactly why the
full-range training/eval must be synthetic. Default AA-only (~1–2 min); add 'SS'/'3Di' to `PROFILE_FEEDS`
to contrast (they are much fatter).

In [ ]:
PROFILE_FEEDS = ['AA']   # add 'SS','3Di' to profile them too (each is another exhaustive pass)
def exhaustive_normlev_hist(feed, block=512, nbins=20, keep_ex=5):
    seqs = POOL_SEQ[feed]; ids = POOL_IDS[feed]; lens = POOL_LEN[feed]; N = len(seqs)
    edges = np.linspace(0, 1, nbins+1); counts = np.zeros(nbins, dtype=np.int64)
    examples = {b: [] for b in range(nbins)}; total = N*(N-1)//2
    for r0 in range(0, N, block):
        r1 = min(r0+block, N)
        D = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - D/den
        for a in range(r1-r0):
            i = r0+a
            if i+1 >= N: continue
            vals = sim[a, i+1:]; js = np.arange(i+1, N)
            bb = np.clip(np.digitize(vals, edges)-1, 0, nbins-1)
            counts += np.bincount(bb, minlength=nbins)
            for b in range(nbins):
                if len(examples[b]) >= keep_ex: continue
                for h in np.where(bb == b)[0][:keep_ex-len(examples[b])]:
                    j = int(js[h]); examples[b].append(dict(domain_a=ids[i], domain_b=ids[j],
                        normLev=round(float(vals[h]), 4), len_a=int(lens[i]), len_b=int(lens[j])))
    dist = pd.DataFrame([dict(bin=f'[{edges[b]:.2f},{edges[b+1]:.2f})', lo=float(edges[b]),
                              count=int(c), percent=100*c/total) for b, c in enumerate(counts)])
    ex = pd.concat([pd.DataFrame(v) for v in examples.values() if v], ignore_index=True) if any(examples.values()) else pd.DataFrame()
    return dist, ex, total

for feed in PROFILE_FEEDS:
    dist, ex, total = exhaustive_normlev_hist(feed)
    print('='*68); print(f'EXHAUSTIVE {feed} normLev distribution — ALL {total:,} unordered pool pairs'); print('='*68)
    print(dist[['bin', 'count', 'percent']].to_string(index=False))
    print(f'\n  pairs >= 0.70: {int(dist[dist.lo >= 0.70]["count"].sum()):,}  |  '
          f'pairs in [0.40,0.70): {int(dist[(dist.lo >= 0.40) & (dist.lo < 0.70)]["count"].sum()):,} (the empty middle)')
    dist.to_csv(f'colab29b_exhaustive_{feed}_normlev_dist.csv', index=False)
    ex.to_csv(f'colab29b_exhaustive_{feed}_examples.csv', index=False)
    fig, ax = plt.subplots(figsize=(10, 4.4))
    ax.bar(np.arange(len(dist)), dist['count'].values, color='#1f77b4')
    ax.set_yscale('symlog'); ax.set_xticks(np.arange(len(dist)))
    ax.set_xticklabels([f'{lo:.2f}' for lo in dist['lo']], rotation=45, ha='right')
    ax.set_xlabel('normLev bin (lower edge)'); ax.set_ylabel('pair count (symlog)')
    _mid = ' — the middle is empty' if feed == 'AA' else ''   # 'empty middle' is an AA-specific claim
    ax.set_title(f'Natural CATH {feed}: exhaustive normLev distribution ({total:,} pairs){_mid}')
    despine(ax); plt.tight_layout()
    plt.savefig(f'colab29b_exhaustive_{feed}_dist.png', dpi=150, bbox_inches='tight'); plt.show()
print('\nData-difficulty slide: natural AA cannot support a full-range correlation analysis ->'
      ' synthetic perturbation is REQUIRED to study the whole edit-distance function.')

## 11c. Slide 17 figure — "Why not just use ESM-2?" (head-to-head, no k-mer baselines)

The same numbers as §9/§10, restricted to the methods the slide compares. This remains CATH-only because
retrieval is only defined for the real full-pool oracle; synthetic stays in the Spearman/AUROC tables.


In [ ]:
# --- Slide 17: SNN vs ESM2 head-to-head (AUROC | MAP@10), values on bars ---
FOCUS_METHODS = ['SNN', 'ESM2', 'ProtTrans', 'length']   # two PLMs (ESM2 + ProtTrans) vs SNN
FOCUS = [m for m in FOCUS_METHODS if m in ACTIVE]
FOCUS_BAR = 0.70                            # retrieval bar shown on the slide

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
x = np.arange(len(FEEDS)); w = 0.8/len(FOCUS)

# panel A — AUROC (full-pool, high >=0.70 vs random negative)
for k, m in enumerate(FOCUS):
    vals = [float(AUROC_TAB.loc[m, f]) for f in FEEDS]
    b = axes[0].bar(x + (k-(len(FOCUS)-1)/2)*w, vals, w, label=m, color=COLORS.get(m, None))
    axes[0].bar_label(b, fmt='%.2f', fontsize=9, padding=2)
axes[0].axhline(0.5, ls='--', color='grey', lw=1)
axes[0].set_ylim(0, 1.12); axes[0].set_ylabel('AUROC (is-high >= 0.70)')
axes[0].set_title('Pairwise discrimination (full-pool)')

# panel B — MAP@10 at FOCUS_BAR (set-based exact-Levenshtein oracle)
sub = retr[retr.bar == FOCUS_BAR]
for k, m in enumerate(FOCUS):
    vals = [float(sub[(sub.method == m) & (sub.feed == f)]['map'].values[0]) for f in FEEDS]
    b = axes[1].bar(x + (k-(len(FOCUS)-1)/2)*w, vals, w, label=m, color=COLORS.get(m, None))
    axes[1].bar_label(b, fmt='%.2f', fontsize=9, padding=2)
axes[1].set_ylim(0, 1.12); axes[1].set_ylabel(f'MAP@10 (set-based, bar {FOCUS_BAR:.2f})')
axes[1].set_title('Set-based retrieval (full-pool)')

for ax in axes:
    ax.set_xticks(x); ax.set_xticklabels(FEEDS); ax.legend(fontsize=9, loc='lower left'); despine(ax)
plt.suptitle('Why not just use a protein LM (ESM-2 / ProtTrans)?  Same pool, same oracle', y=1.02, fontsize=12)
plt.tight_layout(); plt.savefig('colab29b_why_not_plms.png', dpi=150, bbox_inches='tight'); plt.show()

# receipt for the slide caption: the CIs the bars no longer draw
print(f'MAP@10 @ {FOCUS_BAR:.2f} with 95% bootstrap CI (paste into the slide caption):')
for f in FEEDS:
    for m in FOCUS:
        r = sub[(sub.method == m) & (sub.feed == f)].iloc[0]
        print(f'  {f:<4} {m:<6} MAP {r["map"]:.3f}  [{r["map_lo"]:.3f}, {r["map_hi"]:.3f}]   hit@10 {r["hit"]:.3f}')


## 11d. Slide 19 — the FOUR sets side by side (sizes · lengths · letter frequencies · entropy)

**Supervisor ask (2026-07-14):** *"wie gross sind die 4 sets? 4 Verteilungen fuer die Laengen, 4 Verteilungen
fuer die Frequenzen der Buchstaben. Es muss klar ersichtlich sein, inwieweit sich diese aehneln oder
unterscheiden."*

Puts the **synthetic training set** next to the three **real evaluation feeds** on one page:
size, length distribution, per-letter frequency, and Shannon entropy of the letter distribution.

This is the empirical backing for the "why uniform letters" justification: the encoder is trained on a
**maximum-entropy** alphabet and evaluated on three **highly non-uniform** ones — and it still transfers.
It is also the plot that finally licenses talking about 3Di's letter skew (until now that claim was banned
for lack of a frequency plot).


In [ ]:
# --- Slide 19: the four sets — size, length, letter frequency, entropy ---
SYN_N = 3000   # synthetic seed+partner strings to profile (same generator as training)

# rebuild a synthetic sample with the TRAINING generator (uniform AA, same length range, same perturbation)
_rng = np.random.default_rng(0); syn = []
while len(syn) < SYN_N:
    sd = rand_seq(AA_ALPHABET); L = len(sd)
    t = float(_rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
    o = perturb(sd, k, AA_ALPHABET)
    if 1 <= len(o) <= MAX_LEN: syn += [sd, o]
syn = syn[:SYN_N]

SETS = {'synthetic (train)': syn, 'AA (eval)': POOL_SEQ['AA'],
        'SS (eval)': POOL_SEQ['SS'], '3Di (eval)': POOL_SEQ['3Di']}
SETCOL = {'synthetic (train)': '#ff7f0e', 'AA (eval)': '#1f77b4',
          'SS (eval)': '#2ca02c', '3Di (eval)': '#9467bd'}

def letter_freq(seqs, alphabet=AA_ALPHABET):
    c = np.zeros(len(alphabet)); idx = {ch: i for i, ch in enumerate(alphabet)}
    for s in seqs:
        for ch in s:
            if ch in idx: c[idx[ch]] += 1
    return c / max(c.sum(), 1)

def entropy(p):
    p = p[p > 0]; return float(-(p*np.log2(p)).sum())

# ---- summary table ----
print('=' * 88)
print('THE FOUR SETS')
print('=' * 88)
rows = []
for name, seqs in SETS.items():
    L = np.array([len(s) for s in seqs]); f = letter_freq(seqs)
    rows.append(dict(set=name, n_seqs=len(seqs), alphabet_used=int((f > 0).sum()),
                     len_min=int(L.min()), len_med=int(np.median(L)), len_max=int(L.max()),
                     len_mean=round(float(L.mean()), 1), entropy_bits=round(entropy(f), 2),
                     max_entropy=round(np.log2(max((f > 0).sum(), 1)), 2)))
setinfo = pd.DataFrame(rows)
print(setinfo.to_string(index=False))
setinfo.to_csv('colab29b_four_sets.csv', index=False)
print('\nEntropy = Shannon entropy of the letter distribution (bits). "max" = log2(#letters used),')
print('i.e. what a perfectly uniform alphabet of that size would score. Gap = how skewed the set is.')

# ---- figure: lengths (top) + letter frequencies (bottom) ----
fig, axes = plt.subplots(2, 4, figsize=(17, 7.5))
for k, (name, seqs) in enumerate(SETS.items()):
    L = np.array([len(s) for s in seqs]); f = letter_freq(seqs); col = SETCOL[name]
    # top row — length distribution (shared x so the four are comparable at a glance)
    ax = axes[0, k]
    ax.hist(L, bins=30, range=(0, MAX_LEN + 20), color=col, alpha=0.85)
    ax.axvline(L.mean(), ls='--', color='black', lw=1)
    ax.set_title(f'{name}\nn = {len(seqs):,}   len {L.min()}-{L.max()} (mean {L.mean():.0f})', fontsize=10)
    ax.set_xlabel('sequence length'); despine(ax)
    if k == 0: ax.set_ylabel('sequences')
    # bottom row — letter frequency over the 20-letter AA alphabet
    ax = axes[1, k]
    ax.bar(np.arange(len(AA_ALPHABET)), f, color=col)
    ax.axhline(1/20, ls=':', color='grey', lw=1)          # uniform reference over 20 letters
    ax.set_xticks(np.arange(len(AA_ALPHABET)))
    ax.set_xticklabels(list(AA_ALPHABET), fontsize=7)
    used = int((f > 0).sum())
    ax.set_title(f'{used} letters used   H = {entropy(f):.2f} bits '
                 f'(max {np.log2(max(used,1)):.2f})', fontsize=9)
    ax.set_ylim(0, max(0.42, f.max()*1.15)); ax.set_xlabel('letter'); despine(ax)
    if k == 0: ax.set_ylabel('frequency')
plt.suptitle('Training vs evaluation: the encoder is trained on a UNIFORM alphabet and evaluated on three '
             'highly non-uniform ones', y=1.01, fontsize=12)
plt.tight_layout(); plt.savefig('colab29b_four_sets.png', dpi=150, bbox_inches='tight'); plt.show()

print('\nREAD THIS OFF THE FIGURE (for the slide):')
print('  - LENGTHS are matched by construction (all sets live in [%d, %d]) -> length is NOT the confound.' % (MIN_LEN, MAX_LEN))
print('  - LETTERS are not: synthetic is flat (max entropy); AA/3Di are skewed; SS uses only 3 letters.')
print('  - => the encoder never saw natural letter statistics, yet transfers. It learned the OPERATION.')


## 12. How to read this notebook

- **Centerpiece = §8 + §8b Spearman table** (`colab29_spearman.csv/png`). Rows are ordered as
  **SNN / ESM2 / ProtTrans / Dice / trigram / length**; columns are **AA synth / 3Di / SS / AA CATH_s20**. AA synth is
  the in-distribution reference, SS/3Di are the transfer columns, and AA CATH_s20 is the redundancy-reduced
  low-similarity control.
- **Do not grey out AA CATH_s20. Explain it.** CATH_s20 is sequence-redundancy reduced, so high AA global
  similarity is deliberately rare: the exhaustive AA distribution has only a handful of >=0.70 pairs and an
  almost empty middle. AA Spearman mainly measures fine ordering among mutually low-similarity proteins, not
  the retrieval task.
- **Synthetic AUROC is pair-wise; CATH AUROC is full-pool.** This is intentional. The synthetic examples are
  seed+perturbation pairs and do not form an honest all-vs-all retrieval neighbourhood. Say this in the
  caption if synth appears in the AUROC table.
- **Retrieval metric split (§10):** report **AA CATH as hit@10** (pair-like, one true partner), **SS/3Di as
  MAP@10** (many valid neighbours; hit@10 is too forgiving there). Set-based win is real: state the metric
  ("set-based exact-Levenshtein oracle") every time.
- **§11b is the data-distribution defence** (`colab29_exhaustive_AA_*`): natural AA cannot support a
  full-range correlation analysis, which is why synthetic training/evaluation exists as a separate reference.
- **Diagnostic scatter matrix:** `colab29_method_pair_scatter.png` samples the Spearman pair sets to expose
  possible confounds. It is useful for Q&A/debugging, not a headline metric.
- **Ground truth is normLev only.** Fenoy's 0.66 (vs BLASTp identity) is motivation and outlook context,
  never a direct comparison.
- **New-run hygiene:** every artifact uses the `colab29b_*` prefix (does not overwrite the July 16 colab29
  files); the 30k synthetic training set is regenerated each run (seeded, not persisted) and the SNN is
  retrained — note this in the dated `RESULTS_colab29b_*` receipt so an SNN delta is attributable.
- **ProtTrans is LIVE here** (`ENABLE_PROTTRANS=True`, §1) as a second PLM next to ESM-2, pooled to
  parity (trailing `</s>` trimmed, U/Z/O/B→X, cached to `CACHE`). To add any other method: append one entry to
  `METHODS` with a `pair` and `query` function; all tables + figures pick it up automatically.
